# OSM Geofabrik: contenedor PBF de Colombia

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Extracto nacional de OpenStreetMap en PBF. Es el contenedor de origen de capas OSM, no una fuente independiente de evidencia.

In [ ]:
import osmium
PATH=ROOT/"data/cache/colombia-261112.osm.pbf"
meta=ficha(PATH)
reader=osmium.io.Reader(str(PATH),osmium.osm.NOTHING)
try:
 header=reader.header()
 datos_header={"replicacion_utc":header.get("osmosis_replication_timestamp"),"generador":header.get("generator"),"bbox_header":str(header.box())}
finally:reader.close()
display(pd.Series({**meta,**datos_header}).to_frame("valor"))
resumen={"dataset":"12_pbf_colombia",**meta,**datos_header,"filas":None,
         "alcance_eda":"cabecera y huella completas; no se contaron todos los nodos/ways/relaciones"}

## 2. Contrato de extracción disponible en el proyecto

In [ ]:
# No ejecutar descarga ni regeneración: se revisa la configuración del extractor.
import ast
script=ROOT/"scripts/extraer_osm_pbf.py"
arbol=ast.parse(script.read_text())
config={}
for node in arbol.body:
 if isinstance(node,ast.Assign):
  for target in node.targets:
   if isinstance(target,ast.Name) and target.id in {"BBOX","CAPAS"}:
    config[target.id]=ast.literal_eval(node.value)
display(pd.Series({"bbox_cali_sur_oeste_norte_este":config["BBOX"],"capas":list(config["CAPAS"])}).to_frame("valor"))
print("El archivo cubre Colombia; la extracción actual recorta por bbox de Cali.")
print("Ya existen las capas: no se regeneran ni se altera su corte durante este EDA.")
resumen["extraccion_configurada"]={"bbox":config["BBOX"],"capas":list(config["CAPAS"])}

## Reglas decididas para plata

- **Conservar el PBF en caché como respaldo de bronce**, junto con huella, cabecera, corte y configuración de extracción.
- Materializar solo las entidades y columnas necesarias para plata; no subir 319 MiB a Databricks sin una necesidad analítica concreta.
- Para comparar otra ciudad colombiana habrá que parametrizar el área y conservar el mismo corte y filtros. Colombia no cubre Quito ni Santiago de los Caballeros.
- La ausencia de un objeto en una extracción es una propiedad de cobertura y filtros, no una prueba de inexistencia real.
- Este notebook verifica el contenedor; los perfiles tabulares completos corresponden a los notebooks de cada capa materializada.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('12_pbf_colombia', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")